# Fama-French factors for China A-shares

This notebook builds daily **Mkt_RF**, **SMB** and **HML** factors for China A-shares and asks one question:

> How much of the Chinese size premium comes from the smallest stocks?

Liu, Stambaugh and Yuan (2019, *Size and Value in China*) argue that the smallest 30% of Chinese stocks are priced partly as reverse-merger shells, not as operating businesses. To test this, the factors are built twice: on all stocks (standard FF3) and after dropping the smallest 30% (the LSY filter).

## 1. Setup

Prices and the risk-free rate are downloaded with AkShare and cached. Book equity and share counts come from two WIND exports in `data/` (see `data/README.md`). The first run downloads several thousand price histories and takes hours; later runs read the cache.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Make the package importable when the notebook is run from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from ff3_china import Config, run_pipeline
from ff3_china.report import compare_variants, summary_stats

pd.options.display.float_format = "{:.4f}".format
FIGURES = ROOT / "output" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

## 2. Method

| Step | Choice |
|---|---|
| Universe | Shanghai and Shenzhen A-shares, excluding special-treatment (ST) stocks |
| Rebalance | Last trading day of June, each year |
| Size | Unadjusted close x total shares at the rebalance date |
| Book-to-market | Book equity of fiscal year t-1 / size |
| Sorts | Size median (S, B) x B/M 30th and 70th percentiles (L, M, H) |
| Weights | Value-weighted, buy and hold between rebalances |
| Risk-free rate | 3-month ChinaBond government yield |

$$SMB = \tfrac{1}{3}(SL + SM + SH) - \tfrac{1}{3}(BL + BM + BH) \qquad HML = \tfrac{1}{2}(SH + BH) - \tfrac{1}{2}(SL + BL)$$

Both variants share every setting except the filter:

In [ ]:
paths = dict(data_dir=ROOT / "data", cache_dir=ROOT / "cache", output_dir=ROOT / "output")

cfg_ff3 = Config(**paths)                    # all stocks
cfg_lsy = Config(**paths, lsy_filter=True)   # smallest 30% dropped

## 3. Standard FF3

In [ ]:
ff3 = run_pipeline(cfg_ff3)

## 4. With the shell-stock filter

In [ ]:
lsy = run_pipeline(cfg_lsy)

## 5. Results

### Annualised factor premia

`mean_annual` is the average daily return times 252. The t-statistic tests whether the mean differs from zero.

In [ ]:
table = compare_variants({"FF3 (all stocks)": ff3, "LSY (ex smallest 30%)": lsy})
table.style.format({(v, "mean_annual"): "{:.2%}" for v in table.columns.levels[0]} |
                   {(v, "t_stat"): "{:.2f}" for v in table.columns.levels[0]})

In [ ]:
summary_stats(lsy).style.format({
    "mean_annual": "{:.2%}", "vol_annual": "{:.2%}", "sharpe": "{:.2f}",
    "t_stat": "{:.2f}", "min_daily": "{:.2%}", "max_daily": "{:.2%}", "n_obs": "{:.0f}",
}).set_caption("LSY-filtered factors")

### Cumulative factor returns

In [ ]:
FACTORS = ["Mkt_RF", "SMB", "HML"]
STYLES = {"Mkt_RF": "-", "SMB": "--", "HML": ":"}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, (name, data) in zip(axes, {"FF3 (all stocks)": ff3,
                                   "LSY (ex smallest 30%)": lsy}.items()):
    cumulative = (1 + data.set_index("date")[FACTORS]).cumprod() - 1
    for col in FACTORS:
        ax.plot(cumulative.index, cumulative[col], linestyle=STYLES[col], label=col)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_title(name)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.legend()
axes[0].set_ylabel("Cumulative return")
fig.tight_layout()
fig.savefig(FIGURES / "cumulative_factors.png", dpi=150)

### The size premium with and without the smallest stocks

If the smallest stocks carry shell value, SMB built on all stocks should earn more than SMB built on the filtered universe. The gap between the two lines is the part of the size premium that comes from the bottom 30%.

In [ ]:
smb = pd.DataFrame({
    "FF3 (all stocks)": ff3.set_index("date")["SMB"],
    "LSY (ex smallest 30%)": lsy.set_index("date")["SMB"],
}).dropna()
cumulative = (1 + smb).cumprod() - 1

fig, ax = plt.subplots(figsize=(10, 4.5))
for col, style in zip(cumulative.columns, ("-", "--")):
    ax.plot(cumulative.index, cumulative[col], linestyle=style, label=col)
ax.axhline(0, color="grey", linewidth=0.8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_ylabel("Cumulative SMB return")
ax.set_title("Size factor: all stocks vs shell-stock filter")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "smb_comparison.png", dpi=150)

print(f"Annual SMB, all stocks: {smb.iloc[:, 0].mean() * 252:.2%}")
print(f"Annual SMB, filtered:   {smb.iloc[:, 1].mean() * 252:.2%}")
print(f"Daily correlation:      {smb.corr().iloc[0, 1]:.3f}")

## 6. Diagnostics

### Stocks per portfolio

Each of the six portfolios should hold enough stocks every year for its return to be well diversified.

In [ ]:
def portfolio_counts(cfg):
    assignments = pd.read_parquet(cfg.cache_dir / f"assignments_{cfg.cache_tag}.parquet")
    counts = assignments.pivot_table(index="rebal_year", columns="portfolio",
                                     values="code", aggfunc="count")
    counts["Total"] = counts.sum(axis=1)
    return counts

display(portfolio_counts(cfg_ff3).style.set_caption("FF3 (all stocks)"))
display(portfolio_counts(cfg_lsy).style.set_caption("LSY (ex smallest 30%)"))

### Factor correlations

In [ ]:
display(ff3[FACTORS].corr().style.format("{:.3f}").set_caption("FF3 (all stocks)"))
display(lsy[FACTORS].corr().style.format("{:.3f}").set_caption("LSY (ex smallest 30%)"))

### Average return by year

In [ ]:
by_year = lsy.set_index("date")[FACTORS].groupby(lambda d: d.year).mean() * 252
by_year.index.name = "year"
by_year.style.format("{:.2%}").set_caption("LSY-filtered factors, annualised mean by calendar year")

## 7. Limitations

- **Survivorship bias.** The universe is the current exchange listing, so stocks delisted during the sample are missing.
- **ST look-ahead.** Special-treatment status comes from current stock names and is applied to all years.
- **Short sample.** With under ten years of data, premia of a few percent a year are hard to distinguish from zero.
- **Partial LSY replication.** Only the size filter is applied. The original paper also replaces book-to-market with earnings-to-price and rebalances monthly.
- **Total shares.** Market cap uses total shares, including non-tradable shares, not free float.

The factor files are saved in `output/` as `ff3_china_daily.csv` and `ff3_china_daily_lsy.csv`.